## Optional: prepare a sample from 10x files

Convert matched 10x RNA and TCR files into the H5AD input used by creat_graph.py.

Run this notebook from `BTC_demo/notebooks`. Put matched RNA and TCR files from the same cell library in `../examples/raw/`:

```text
../examples/raw/
  barcodes.tsv.gz
  features.tsv.gz
  matrix.mtx.gz
  filtered_contig_annotations.csv.gz
```

Set the actual patient and sample IDs in the configuration cell. The example joins by barcode, selects productive high-confidence TRA/TRB contigs, and chooses the highest-UMI contig per chain (then reads, then contig ID). The default QC thresholds are ≥200 detected genes and ≤20% mitochondrial counts, with at least one usable TCR chain. Set the thresholds in the configuration cell. Mitochondrial genes are identified by the human gene-symbol prefix `MT-`. CDR3 sequences retain the junction boundaries supplied in the input.

Outputs are written to `../outputs/prepared/`; rerunning replaces those preparation outputs. Then run `python creat_graph.py outputs/prepared/sample_graph_input.h5ad --output-dir graphs` from the `BTC_demo` directory.


In [ ]:
from pathlib import Path
import gzip
import json

import anndata as ad
import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp
from scipy.io import mmread

# Input and output paths relative to the notebook working directory.
DATA_DIR = Path("../examples/raw")
OUTPUT_DIR = Path("../outputs/prepared")
RNA_PREFIX = ""  # Set a prefix here if your RNA filenames have one.
TCR_PATH = DATA_DIR / "filtered_contig_annotations.csv.gz"
SAMPLE_ID = "sample_1"
PATIENT_ID = "patient_1"

# QC thresholds for detected genes and mitochondrial count percentage.
MIN_GENES = 200
MAX_MT_PERCENT = 20.0

required_paths = [DATA_DIR / (RNA_PREFIX + suffix) for suffix in
                  ["barcodes.tsv.gz", "features.tsv.gz", "matrix.mtx.gz"]]
required_paths.append(TCR_PATH)
missing = [str(path) for path in required_paths if not path.is_file()]
assert not missing, f"Input files not found; check DATA_DIR and RNA_PREFIX: {missing}"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Input:", DATA_DIR)
print("Output:", OUTPUT_DIR)


In [ ]:
# Read RNA counts and create the cell metadata.
features = pd.read_csv(DATA_DIR / (RNA_PREFIX + 'features.tsv.gz'), sep='\t', header=None, dtype=str, keep_default_na=False)
barcodes = pd.read_csv(DATA_DIR / (RNA_PREFIX + 'barcodes.tsv.gz'), sep='\t', header=None, dtype=str)[0]
assert barcodes.is_unique and not barcodes.isna().any()
with gzip.open(DATA_DIR / (RNA_PREFIX + 'matrix.mtx.gz'), 'rb') as handle:
    matrix = mmread(handle).tocsr()
assert matrix.shape == (len(features), len(barcodes))
assert np.isfinite(matrix.data).all() and (matrix.data >= 0).all()
assert np.equal(matrix.data, np.floor(matrix.data)).all(), 'Expected raw integer counts'

is_gene = features[2].eq('Gene Expression') if features.shape[1] >= 3 else np.ones(len(features), dtype=bool)
genes = features.loc[is_gene].copy()
assert genes[0].is_unique and genes[1].ne('').all()
X = matrix[np.asarray(is_gene)].T.tocsr()
# Sum counts for duplicate gene symbols.
codes, symbols = pd.factorize(genes[1], sort=False)
aggregation = sp.csr_matrix((np.ones(len(genes), dtype=np.int64), (np.arange(len(genes)), codes)), shape=(len(genes), len(symbols)))
X = (X @ aggregation).tocsr()
var = pd.DataFrame(index=pd.Index(symbols, name='gene_symbol'))
var['gene_ids'] = genes.groupby(1, sort=False)[0].agg(';'.join).reindex(symbols).to_numpy()
var['n_source_features'] = np.bincount(codes)

obs = pd.DataFrame(index=pd.Index(SAMPLE_ID + ':' + barcodes, name='cell_id'))
obs['barcode'] = barcodes.to_numpy()
obs['patient_id'] = PATIENT_ID
obs['sample_id'] = SAMPLE_ID
adata = ad.AnnData(X=X, obs=obs, var=var)
adata.uns['X_description'] = 'Raw RNA counts; duplicate gene symbols summed; no normalization.'
print(adata)


In [ ]:
# Match TCR chains to RNA barcodes.
contigs = pd.read_csv(TCR_PATH, keep_default_na=False)
assert contigs['contig_id'].is_unique
for col in ['is_cell', 'high_confidence', 'productive']:
    values = contigs[col].astype(str).str.lower()
    assert values.isin(['true', 'false']).all()
    contigs[col] = values.eq('true')
for col in ['umis', 'reads']:
    contigs[col] = pd.to_numeric(contigs[col], errors='raise')
valid = contigs[['is_cell', 'high_confidence', 'productive']].all(axis=1) & contigs.chain.isin(['TRA', 'TRB']) & contigs.cdr3.str.fullmatch('[ACDEFGHIKLMNPQRSTVWY]+', na=False)
contigs['usable_for_selection'] = valid
contigs['in_rna'] = contigs.barcode.isin(barcodes)
usable = contigs.loc[valid].sort_values(['barcode', 'chain', 'umis', 'reads', 'contig_id'], ascending=[True, True, False, False, True])
selected = usable.drop_duplicates(['barcode', 'chain'])
contigs['selected_primary_chain'] = contigs.contig_id.isin(selected.contig_id)

for chain, suffix in [('TRA', 'a'), ('TRB', 'b')]:
    table = selected.loc[selected.chain.eq(chain)].set_index('barcode').reindex(barcodes)
    fields = {'cdr3': f'cdr3_aa_{suffix}', 'v_gene': f'v_gene_{suffix}', 'j_gene': f'j_gene_{suffix}', 'contig_id': f'contig_id_{suffix}', 'raw_clonotype_id': f'raw_clonotype_id_{suffix}', 'umis': f'tcr_umis_{suffix}', 'reads': f'tcr_reads_{suffix}'}
    if chain == 'TRB': fields['d_gene'] = 'd_gene_b'
    for source, target in fields.items():
        values = table[source].mask(table[source].isin(['', 'None', 'NA'])).to_numpy()
        adata.obs[target] = values if source in ['umis', 'reads'] else pd.Categorical(values)
    counts = usable.loc[usable.chain.eq(chain)].groupby('barcode').size().reindex(barcodes, fill_value=0)
    adata.obs[f'n_{chain}_contigs'] = counts.to_numpy()

adata.obs['has_tcr_contig'] = adata.obs.barcode.isin(contigs.barcode)
adata.obs['has_TRA'] = adata.obs.cdr3_aa_a.notna()
adata.obs['has_TRB'] = adata.obs.cdr3_aa_b.notna()
adata.obs['has_paired_tcr'] = adata.obs.has_TRA & adata.obs.has_TRB
adata.obs['has_multichain_tcr'] = (adata.obs[['n_TRA_contigs', 'n_TRB_contigs']] > 1).any(axis=1)
qc = dict(rna_cells=adata.n_obs, tcr_cells=int(contigs.barcode.nunique()), barcode_overlap=int(adata.obs.has_tcr_contig.sum()), usable_tcr_cells=int((adata.obs.has_TRA | adata.obs.has_TRB).sum()), paired_tcr_cells=int(adata.obs.has_paired_tcr.sum()), multichain_cells=int(adata.obs.has_multichain_tcr.sum()))
print(pd.Series(qc))
assert qc['usable_tcr_cells'] > 0
adata.uns['tcr_selection'] = 'Productive high-confidence cell TRA/TRB with canonical CDR3; choose highest UMI then reads, ties by contig_id.'
adata.uns['pairing_qc'] = qc


In [ ]:
# Filter cells, save H5AD, and check the saved input.
adata.var["mt"] = adata.var_names.str.upper().str.startswith("MT-")
assert adata.var["mt"].any(), "No mitochondrial genes detected; check gene naming."
sc.pp.calculate_qc_metrics(
    adata, qc_vars=["mt"], percent_top=None, inplace=True
)
adata.obs[["total_counts", "n_genes_by_counts", "pct_counts_mt"]].describe(
    percentiles=[0.05, 0.5, 0.95]
)

adata.obs["pass_rna_qc"] = (
    (adata.obs["n_genes_by_counts"] >= MIN_GENES)
    & (adata.obs["pct_counts_mt"] <= MAX_MT_PERCENT)
)
adata.obs["pass_graph_input_qc"] = (
    adata.obs["pass_rna_qc"]
    & (adata.obs["has_TRA"] | adata.obs["has_TRB"])
)
graph_adata = adata[adata.obs["pass_graph_input_qc"]].copy()
assert graph_adata.n_obs > 0, "No cells passed QC; inspect the inputs and thresholds."

qc.update({
    "min_genes": MIN_GENES,
    "max_mitochondrial_percent": MAX_MT_PERCENT,
    "rna_qc_pass_cells": int(adata.obs["pass_rna_qc"].sum()),
    "graph_input_cells": graph_adata.n_obs,
    "graph_input_paired_cells": int(graph_adata.obs["has_paired_tcr"].sum()),
})
adata.uns["preprocessing_qc"] = qc.copy()
graph_adata.uns["preprocessing_qc"] = qc.copy()
print(pd.Series(qc))

all_path = OUTPUT_DIR / f"{SAMPLE_ID}_rna_tcr_all.h5ad"
INPUT_H5AD = OUTPUT_DIR / "sample_graph_input.h5ad"
adata.write_h5ad(all_path, compression="gzip")
graph_adata.write_h5ad(INPUT_H5AD, compression="gzip")
contigs.to_csv(OUTPUT_DIR / f"{SAMPLE_ID}_contig_audit.csv.gz", index=False)
(OUTPUT_DIR / f"{SAMPLE_ID}_pairing_qc.json").write_text(
    json.dumps(qc, indent=2) + "\n"
)
print("All RNA cells:", all_path)
print("Graph input:", INPUT_H5AD)

restored = ad.read_h5ad(INPUT_H5AD)
assert restored.obs_names.equals(graph_adata.obs_names)
assert restored.var_names.equals(graph_adata.var_names)
assert (restored.X != graph_adata.X).nnz == 0
pd.testing.assert_frame_equal(restored.obs, graph_adata.obs)
print("Read-back checks passed.")
print(restored)
